# NB15 — Build the clean, complete **ai-image-detection-dataset**

Reads `Shanmuk4622/ai-detection-dataset-v2` and assembles a **new, self-contained repo**
`Shanmuk4622/ai-image-detection-dataset` where every row is complete on its own:

- **`image` → HF `Image` type** so thumbnails render in the viewer.
- **`prompt` filled on real rows** — each real image's BLIP-2 caption is joined back on.
- **Per-model subsets** (`default`, `real`, `sd15`, `sdxl`, `flux_schnell`, `kandinsky22`,
  `pixart_sigma`, `wuerstchen`), each with **train / validation / test** splits.
- **Caption provenance** on the row (`raw_caption`, `caption_n_tokens`, `caption_model`) **and**
  the original `captions/` shards copied verbatim.
- Side tables consolidated under `metadata/`.

Layout: `<subset>/{train,val,test}/*.parquet`. **PNG bytes copied verbatim.** v2/v3 untouched.

## Commit / push model
- **Commit = local, per source shard** (cheap, offline, continuous).
- **Push = to HF, throttled**: only every **≥20 min**, on a **major cell**, at a **size cap**,
  or when **you stop** — one commit per push (well under HF's ~128/hr limit).
- **Resumable & self-healing**: the ledger `_build_state.json` stamps a `LAYOUT` version and
  auto-resets if the layout changed. Stop = immediate push.

Kaggle: Internet ON · GPU not needed · secret **`HF_TOKEN`** (write) · ~20 GB disk is plenty.


In [ ]:
# 1) deps --------------------------------------------------------------
import importlib.util, subprocess, sys
need=[p for m,p in [("datasets","datasets>=2.19"),("huggingface_hub","huggingface_hub>=0.24"),
                    ("pyarrow","pyarrow"),("PIL","pillow"),("yaml","pyyaml")]
      if importlib.util.find_spec(m) is None]
if need: subprocess.run([sys.executable,"-m","pip","install","-q",*need],check=True); print("installed:",need)
else: print("all deps present")

In [ ]:
# 2) config + token ----------------------------------------------------
import os, io, json, time, base64, shutil, signal, atexit, glob, traceback
import pyarrow as pa, pyarrow.parquet as pq
import yaml
from huggingface_hub import HfApi, hf_hub_download, CommitOperationAdd
from huggingface_hub.utils import HfHubHTTPError
import datasets
from datasets import Dataset, Image

SRC_REPO  = "Shanmuk4622/ai-detection-dataset-v2"       # read-only source
DST_REPO  = "Shanmuk4622/ai-image-detection-dataset"    # NEW repo we build
REPO_TYPE = "dataset"
RESET_BUILD = False   # set True to force a full wipe+rebuild (layout changes auto-reset anyway)

PUSH_EVERY_SEC=20*60; MAX_STAGE_BYTES=6*1024**3; MIN_PUSH_GAP_SEC=45
MAX_PUSH_PER_HR=90; PUSH_RETRIES=6

WORK  = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/tmp/work"
STAGE = os.path.join(WORK,"stage"); DLDIR=os.path.join(WORK,"dl")
LOCAL_LEDGER=os.path.join(WORK,"local_progress.json")
os.makedirs(STAGE,exist_ok=True); os.makedirs(DLDIR,exist_ok=True)

def load_token():
    try:
        from kaggle_secrets import UserSecretsClient
        t=UserSecretsClient().get_secret("HF_TOKEN")
        if t: return t.strip()
    except Exception: pass
    for k in ("HF_TOKEN","HUGGINGFACE_TOKEN","HUGGING_FACE_HUB_TOKEN"):
        if os.environ.get(k): return os.environ[k].strip()
    import getpass; return getpass.getpass("HF write token: ").strip()

TOKEN=load_token(); api=HfApi(token=TOKEN)
print("whoami:",api.whoami()["name"],"|",SRC_REPO,"->",DST_REPO)

In [ ]:
# 3) enriched schema + subset/split maps + state -----------------------
ENRICHED = pa.schema([
    ("image",            pa.struct([("bytes",pa.binary()),("path",pa.string())])),
    ("image_id",         pa.string()),
    ("source_real_id",   pa.string()),
    ("label",            pa.int8()),
    ("generator",        pa.string()),
    ("source_dataset",   pa.string()),
    ("split",            pa.string()),
    ("prompt",           pa.string()),
    ("raw_caption",      pa.string()),
    ("caption_n_tokens", pa.int16()),
    ("caption_model",    pa.string()),
    ("width",            pa.int16()),
    ("height",           pa.int16()),
    ("orig_width",       pa.int32()),
    ("orig_height",      pa.int32()),
    ("gen_model_id",     pa.string()),
    ("gen_steps",        pa.int16()),
    ("gen_guidance",     pa.float32()),
    ("gen_native_res",   pa.int16()),
    ("seed",             pa.int64()),
    ("pipeline_version", pa.string()),
    ("sha256",           pa.string()),
    ("created_utc",      pa.string()),
])
SUBSETS      = ["real","sd15","sdxl","flux_schnell","kandinsky22","pixart_sigma","wuerstchen"]
SPLIT_FOLDER = {"train":"train","val":"val","test":"test"}
SPLIT_NAME   = {"train":"train","val":"validation","test":"test"}
STATE_PATH   = "_build_state.json"
LAYOUT       = "subset-v1"   # bump if the on-repo folder layout changes -> triggers auto-reset

def subset_of(src_path):
    return "real" if src_path.startswith("real/") else src_path.split("/")[1]
def ensure_dst(): api.create_repo(DST_REPO,repo_type=REPO_TYPE,private=True,exist_ok=True)
def load_state():
    existed=True
    try:
        p=hf_hub_download(DST_REPO,STATE_PATH,repo_type=REPO_TYPE,token=TOKEN,force_download=True)
        s=json.load(open(p))
    except Exception: s={}; existed=False
    s.setdefault("pushed_sources",[]); s.setdefault("meta_done",False); s.setdefault("readme_done",False)
    return s, existed
def list_image_shards():
    fs=api.list_repo_files(SRC_REPO,repo_type=REPO_TYPE)
    return sorted(f for f in fs if f.endswith(".parquet") and (f.startswith("real/") or f.startswith("data/")))
print("schema + maps ready")

In [ ]:
# 4) load side tables ONCE: split map + caption map --------------------
def _dl(path): return hf_hub_download(SRC_REPO,path,repo_type=REPO_TYPE,token=TOKEN)
sp = pq.read_table(_dl("splits.parquet"))
split_of = dict(zip(sp.column("source_real_id").to_pylist(), sp.column("split").to_pylist()))
print("split map:", len(split_of), "ids")

cap_files=[f for f in api.list_repo_files(SRC_REPO,repo_type=REPO_TYPE)
           if f.startswith("captions/") and f.endswith(".parquet")]
captions={}
for f in sorted(cap_files):
    t=pq.read_table(_dl(f)); cols={n:t.column(n).to_pylist() for n in t.schema.names}
    for i in range(t.num_rows):
        captions[cols["source_real_id"][i]]={
            "caption":cols["caption"][i] if "caption" in cols else None,
            "raw_caption":cols["raw_caption"][i] if "raw_caption" in cols else None,
            "n_tokens":cols["n_tokens"][i] if "n_tokens" in cols else None,
            "caption_model":cols["caption_model"][i] if "caption_model" in cols else None}
print("caption map:", len(captions), "ids | caption source shards:", len(cap_files))

In [ ]:
# 5) convert ONE source shard -> up to 3 <subset>/<split> parquet files -
def _col(tbl,name,n): return tbl.column(name).to_pylist() if name in tbl.schema.names else [None]*n

def convert_source(src_path):
    sub=subset_of(src_path)
    local=hf_hub_download(SRC_REPO,src_path,repo_type=REPO_TYPE,token=TOKEN,local_dir=DLDIR)
    tbl=pq.read_table(local); n=tbl.num_rows
    img=tbl.column("image").to_pylist()
    g={c:_col(tbl,c,n) for c in ["image_id","source_real_id","label","generator","source_dataset",
        "prompt","caption_model","width","height","orig_width","orig_height","gen_model_id",
        "gen_steps","gen_guidance","gen_native_res","seed","pipeline_version","sha256","created_utc"]}
    buckets={}; miss_split=0; miss_cap=0
    for i in range(n):
        sid=g["source_real_id"][i]; sraw=split_of.get(sid)
        if sraw is None: miss_split+=1; continue
        cap=captions.get(sid,{});  miss_cap+= (0 if cap else 1)
        iid=g["image_id"][i]
        row={"image":{"bytes":img[i],"path":(f"{iid}.png" if iid else None)},
            "image_id":iid,"source_real_id":sid,"label":g["label"][i],
            "generator":g["generator"][i],"source_dataset":g["source_dataset"][i],
            "split":SPLIT_NAME[sraw],
            "prompt":(g["prompt"][i] or cap.get("caption")),
            "raw_caption":cap.get("raw_caption"),"caption_n_tokens":cap.get("n_tokens"),
            "caption_model":(g["caption_model"][i] or cap.get("caption_model")),
            "width":g["width"][i],"height":g["height"][i],
            "orig_width":g["orig_width"][i],"orig_height":g["orig_height"][i],
            "gen_model_id":g["gen_model_id"][i],"gen_steps":g["gen_steps"][i],
            "gen_guidance":g["gen_guidance"][i],"gen_native_res":g["gen_native_res"][i],
            "seed":g["seed"][i],"pipeline_version":g["pipeline_version"][i],
            "sha256":g["sha256"][i],"created_utc":g["created_utc"][i]}
        buckets.setdefault(SPLIT_FOLDER[sraw],[]).append(row)
    base=os.path.basename(src_path); staged=[]
    for folder,rows in buckets.items():
        t=pa.Table.from_pylist(rows,schema=ENRICHED)
        ds=Dataset(t).cast_column("image",Image())
        repo_path=f"{sub}/{folder}/{base}"; out=os.path.join(STAGE,repo_path)
        os.makedirs(os.path.dirname(out),exist_ok=True); ds.to_parquet(out)
        staged.append((repo_path,out))
    try: os.remove(local)
    except OSError: pass
    return staged, n, miss_split, miss_cap
print("convert_source ready")

In [ ]:
# 6) Pusher: local commits per source; push to HF only on triggers -----
class Pusher:
    def __init__(self,state):
        self.state=state; self.items=[]; self.last_push=time.time(); self.push_times=[]
    def commit_local(self,src,files,rows):
        self.items.append({"src":src,"files":files})
        json.dump({"staged_sources":[it["src"] for it in self.items],
                   "pushed_sources":self.state["pushed_sources"]},open(LOCAL_LEDGER,"w"),indent=2)
        mins=max(0,(PUSH_EVERY_SEC-(time.time()-self.last_push)))/60
        print(f"    committed locally: {src} ({rows} rows -> {len(files)} split file(s))"
              f" [staged {len(self.items)} src, {self.bytes()/1e9:.2f} GB, next push ~{mins:.1f} min]")
    def bytes(self): return sum(os.path.getsize(l) for it in self.items for _,l in it["files"] if os.path.exists(l))
    def due(self): return self.items and ((time.time()-self.last_push)>=PUSH_EVERY_SEC or self.bytes()>=MAX_STAGE_BYTES)
    def _rl(self):
        now=time.time(); self.push_times=[t for t in self.push_times if now-t<=3600]
        if len(self.push_times)>=MAX_PUSH_PER_HR:
            w=3600-(now-self.push_times[0])+5; print(f"    [rate-limit] sleeping {w:.0f}s"); time.sleep(max(w,0))
    def push(self,reason,force=False):
        if not self.items: return
        if not force and (time.time()-self.last_push)<MIN_PUSH_GAP_SEC: return
        ops=[]; srcs=[]
        for it in self.items:
            srcs.append(it["src"])
            for rp,l in it["files"]: ops.append(CommitOperationAdd(path_in_repo=rp,path_or_fileobj=l))
        pushed=sorted(set(self.state["pushed_sources"])|set(srcs)); st=dict(self.state); st["pushed_sources"]=pushed
        ops.append(CommitOperationAdd(STATE_PATH,json.dumps(st,indent=2).encode()))
        print(f"  >> PUSH: {len(srcs)} source shard(s), {len(ops)-1} file(s) [{reason}, {self.bytes()/1e9:.2f} GB]")
        self._rl()
        for a in range(1,PUSH_RETRIES+1):
            try:
                api.create_commit(DST_REPO,repo_type=REPO_TYPE,operations=ops,
                                  commit_message=f"build: +{len(srcs)} source shards ({reason})"); break
            except HfHubHTTPError as e:
                b=min(60*a,300); print(f"     retry {a}/{PUSH_RETRIES} http={getattr(getattr(e,'response',None),'status_code',None)}: {b}s"); time.sleep(b)
                if a==PUSH_RETRIES: raise
            except Exception as e:
                b=min(30*a,180); print(f"     retry {a}/{PUSH_RETRIES} {type(e).__name__}: {b}s"); time.sleep(b)
                if a==PUSH_RETRIES: raise
        self.state["pushed_sources"]=pushed; self.push_times.append(time.time()); self.last_push=time.time()
        for it in self.items:
            for _,l in it["files"]:
                try: os.remove(l)
                except OSError: pass
        self.items=[]; json.dump({"staged_sources":[],"pushed_sources":pushed},open(LOCAL_LEDGER,"w"),indent=2)
        shutil.rmtree(STAGE,ignore_errors=True); os.makedirs(STAGE,exist_ok=True)
        shutil.rmtree(os.path.expanduser("~/.cache/huggingface/hub"),ignore_errors=True)

PUSHER=None
def _sig(sig,frm):
    print(f"\n[signal {sig}] STOP -> immediate push ...")
    try:
        if PUSHER: PUSHER.push("interrupt",force=True)
    finally: raise KeyboardInterrupt
for _s in (signal.SIGINT,signal.SIGTERM):
    try: signal.signal(_s,_sig)
    except Exception: pass
atexit.register(lambda: PUSHER and PUSHER.push("final",force=True))
print("pusher ready")

In [ ]:
# 7) one-time: (auto/forced reset), create repo, metadata/, captions/, card
ensure_dst()
STATE, existed = load_state()
need_reset = RESET_BUILD or (existed and STATE.get("layout") != LAYOUT)
if need_reset:
    why = "RESET_BUILD=True" if RESET_BUILD else f"layout change ({STATE.get('layout')!r} -> {LAYOUT!r})"
    print("resetting DST because:", why)
    for d in ["data","train","val","test","captions","metadata",*SUBSETS]:
        try: api.delete_folder(path_in_repo=d,repo_id=DST_REPO,repo_type=REPO_TYPE,commit_message=f"reset {d}")
        except Exception: pass
    for froot in ["config.json","manifest.parquet","splits.parquet","validation_report.json",STATE_PATH]:
        try: api.delete_file(froot,repo_id=DST_REPO,repo_type=REPO_TYPE)
        except Exception: pass
    STATE={"pushed_sources":[],"meta_done":False,"readme_done":False}
    print("DST wiped — rebuilding fresh in layout", LAYOUT)
STATE["layout"]=LAYOUT
PUSHER=Pusher(STATE)
print("resuming; source shards already pushed:",len(STATE["pushed_sources"]))

if not STATE["meta_done"]:
    ops=[]
    for src,dst in [("config.json","metadata/config.json"),
                    ("validation_report.json","metadata/validation_report.json"),
                    ("splits.parquet","metadata/splits.parquet"),
                    ("manifest.parquet","metadata/manifest.parquet"),
                    ("ai_detect_common.py","ai_detect_common.py")]:
        try: ops.append(CommitOperationAdd(dst,hf_hub_download(SRC_REPO,src,repo_type=REPO_TYPE,token=TOKEN)))
        except Exception as e: print("  skip",src,"->",e)
    try:
        cf=[f for f in api.list_repo_files(SRC_REPO,repo_type=REPO_TYPE)
            if f.startswith("captions/") and f.endswith(".parquet")]
        tabs=[]
        for f in sorted(cf):
            lp=hf_hub_download(SRC_REPO,f,repo_type=REPO_TYPE,token=TOKEN)
            ops.append(CommitOperationAdd(f,lp)); tabs.append(pq.read_table(lp))
        if tabs:
            cap_all=pa.concat_tables(tabs,promote_options="default")
            cp=os.path.join(WORK,"captions.parquet"); pq.write_table(cap_all,cp,compression="zstd")
            ops.append(CommitOperationAdd("metadata/captions.parquet",cp))
            print("captions:",len(cf),"shards +",cap_all.num_rows,"consolidated rows")
    except Exception as e: print("  captions copy skipped:",e)
    STATE["meta_done"]=True
    ops.append(CommitOperationAdd(STATE_PATH,json.dumps(STATE,indent=2).encode()))
    api.create_commit(DST_REPO,repo_type=REPO_TYPE,operations=ops,commit_message="build: metadata/ + captions/")
    print("pushed metadata/ + captions/")

if not STATE["readme_done"]:
    configs=[{"config_name":"default","data_files":[
        {"split":"train","path":[f"{s}/train/*.parquet" for s in SUBSETS]},
        {"split":"validation","path":[f"{s}/val/*.parquet" for s in SUBSETS]},
        {"split":"test","path":[f"{s}/test/*.parquet" for s in SUBSETS]}]}]
    for s in SUBSETS:
        configs.append({"config_name":s,"data_files":[
            {"split":"train","path":f"{s}/train/*.parquet"},
            {"split":"validation","path":f"{s}/val/*.parquet"},
            {"split":"test","path":f"{s}/test/*.parquet"}]})
    front={"license":"other","task_categories":["image-classification"],"language":["en"],
        "tags":["ai-generated-image-detection","synthetic-image-detection","diffusion-models","image-forensics"],
        "pretty_name":"AI-Image Detection Dataset","size_categories":["10K<n<100K"],"configs":configs}
    body=base64.b64decode("IyBBSS1JbWFnZSBEZXRlY3Rpb24gRGF0YXNldAoKKipQYWlyZWQgcmVhbCAvIEFJIGltYWdlcywgd2l0aCBzaGFyZWQgaW1hZ2UtZ3JvdW5kZWQgY2FwdGlvbnMsIGZvciB0cmFpbmluZyBhbmQKZXZhbHVhdGluZyBBSS1nZW5lcmF0ZWQtaW1hZ2UgZGV0ZWN0b3JzLioqCgpFYWNoIG9mIDEwLDAwMCByZWFsIHBob3RvcyBpcyBjYXB0aW9uZWQgb25jZSAoQkxJUC0yKSBhbmQgcGFpcmVkIHdpdGggKipvbmUgc3ludGhldGljCnBhcnRuZXIgcGVyIGdlbmVyYXRvcioqICg2IGdlbmVyYXRvcnMg4oaSIDYwLDAwMCBBSSBpbWFnZXMpLiBBIHJlYWwgaW1hZ2UgYW5kIGFsbCBvZiBpdHMKQUkgcGFydG5lcnMgKipzaGFyZSB0aGUgc2FtZSBwcm9tcHQqKiwgc28gdGhlICpvbmx5KiBzeXN0ZW1hdGljIGRpZmZlcmVuY2UgYmV0d2VlbiB0aGUKY2xhc3NlcyBpcyB0aGUgZ2VuZXJhdGl2ZSBwcm9jZXNzIGl0c2VsZi4gQSBkZXRlY3RvciB0cmFpbmVkIGhlcmUgaXMgcHVzaGVkIHRvd2FyZCB0aGUKKipzeW50aGVzaXMgZmluZ2VycHJpbnQqKiAodGV4dHVyZSwgZnJlcXVlbmN5LCByZW5kZXJpbmcgYXJ0ZWZhY3RzKSBpbnN0ZWFkIG9mIHNjZW5lCmNvbnRlbnQg4oCUIHRoZSBzaG9ydGN1dCB0aGF0IGluZmxhdGVzIGFjY3VyYWN5IG9uIG5haXZlbHktYnVpbHQgZGF0YXNldHMuCgo+IFRoaXMgaXMgdGhlIGNsZWFuLCBzZWxmLWNvbnRhaW5lZCByZWxlYXNlLiBFdmVyeSByb3cgaXMgY29tcGxldGUgb24gaXRzIG93bjogdGhlCj4gKippbWFnZSByZW5kZXJzIGluIHRoZSB2aWV3ZXIqKiwgY2FycmllcyBpdHMgKipwcm9tcHQqKiwgaXRzICoqdHJhaW4vdmFsL3Rlc3Qgc3BsaXQqKiwKPiBhbmQgZnVsbCAqKmNhcHRpb24gKyBnZW5lcmF0aW9uIHByb3ZlbmFuY2UqKiDigJQgbm8gbmVlZCB0byBqb2luIGFnYWluc3Qgc2lkZSBmaWxlcy4gVGhlCj4gdmlld2VyIGV4cG9zZXMgYSAqKmBkZWZhdWx0YCBzdWJzZXQgKGFsbCA3MGspKiogcGx1cyAqKm9uZSBzdWJzZXQgcGVyIG1vZGVsKiosIGVhY2ggd2l0aAo+IGl0cyBvd24gdHJhaW4vdmFsaWRhdGlvbi90ZXN0IHNwbGl0cy4KCiMjIEF0IGEgZ2xhbmNlCgp8IHwgfAp8LS0tfC0tLXwKfCBSZWFsIGltYWdlcyB8IDEwLDAwMCAoQ09DTyArIEltYWdlTmV0LTFrKSwgbGFiZWwgYDBgIHwKfCBBSSBpbWFnZXMgfCA2MCwwMDAgYWNyb3NzIDYgZ2VuZXJhdG9ycywgbGFiZWwgYDFgIHwKfCBUb3RhbCByb3dzIHwgNzAsMDAwIHwKfCBQYWlycyB8IDEwLDAwMCAoZWFjaCByZWFsIOKGlCA2IEFJIHBhcnRuZXJzLCBzaGFyZWQgY2FwdGlvbikgfAp8IFJlc29sdXRpb24gfCA1MTLDlzUxMiBSR0IgUE5HIHwKfCBTcGxpdHMgfCB0cmFpbiA0OSwzOTIgLyB2YWxpZGF0aW9uIDEwLDEyMiAvIHRlc3QgMTAsNDg2IChwYWlyLWxldmVsLCBzZWVkIDQyKSB8CnwgU3Vic2V0cyB8IGBkZWZhdWx0YCArIGByZWFsYCwgYHNkMTVgLCBgc2R4bGAsIGBmbHV4X3NjaG5lbGxgLCBga2FuZGluc2t5MjJgLCBgcGl4YXJ0X3NpZ21hYCwgYHd1ZXJzdGNoZW5gIHwKfCBQcmVwcm9jZXNzaW5nIHwgSWRlbnRpY2FsIGNhbm9uaWNhbCBwaXBlbGluZSBmb3IgcmVhbCAqKmFuZCoqIEFJIChgcGlwZWxpbmVfdmVyc2lvbiAxLjJgKSB8CnwgTGljZW5zZSB8IE5vbi1jb21tZXJjaWFsIHJlc2VhcmNoIHVzZSAobW9zdC1yZXN0cmljdGl2ZSBpbmhlcml0ZWQgdGVybSkgfAoKIyMgU3Vic2V0cwoKVGhlIERhdGFzZXQgVmlld2VyIGFuZCBgbG9hZF9kYXRhc2V0YCBleHBvc2UgZWlnaHQgc3Vic2V0cywgZWFjaCBzcGxpdCBpbnRvCnRyYWluIC8gdmFsaWRhdGlvbiAvIHRlc3Q6Cgp8IHN1YnNldCB8IHJvd3MgfCBjb250ZW50cyB8CnwtLS0tLS0tLXwtLS0tLS18LS0tLS0tLS0tLXwKfCBgZGVmYXVsdGAgfCA3MCwwMDAgfCBldmVyeXRoaW5nIChyZWFsICsgYWxsIGdlbmVyYXRvcnMpIHwKfCBgcmVhbGAgfCAxMCwwMDAgfCByZWFsIHBob3RvcyBvbmx5IChsYWJlbCAwKSB8CnwgYHNkMTVgIHwgMTAsMDAwIHwgU3RhYmxlIERpZmZ1c2lvbiAxLjUgb3V0cHV0cyB8CnwgYHNkeGxgIHwgMTAsMDAwIHwgU0RYTCBvdXRwdXRzIHwKfCBgZmx1eF9zY2huZWxsYCB8IDEwLDAwMCB8IEZMVVguMS1zY2huZWxsIG91dHB1dHMgfAp8IGBrYW5kaW5za3kyMmAgfCAxMCwwMDAgfCBLYW5kaW5za3kgMi4yIG91dHB1dHMgfAp8IGBwaXhhcnRfc2lnbWFgIHwgMTAsMDAwIHwgUGl4QXJ0Lc6jIG91dHB1dHMgfAp8IGB3dWVyc3RjaGVuYCB8IDEwLDAwMCB8IFfDvHJzdGNoZW4gb3V0cHV0cyB8CgojIyBXaGF0IGNoYW5nZWQgdnMuIHRoZSBlYXJsaWVyIHJlbGVhc2UKCi0gKipgaW1hZ2VgIGlzIG5vdyB0aGUgSHVnZ2luZyBGYWNlIGBJbWFnZWAgdHlwZSoqIOKGkiB0aHVtYm5haWxzIHJlbmRlciBpbiB0aGUgRGF0YXNldCBWaWV3ZXIuCi0gKipgcHJvbXB0YCBpcyBwb3B1bGF0ZWQgb24gcmVhbCByb3dzKiogKGVhY2ggcmVhbCBpbWFnZSdzIG93biBCTElQLTIgY2FwdGlvbiksIHNvIGEgcGFpcgogIGxpdGVyYWxseSBzaGFyZXMgb25lIGNhcHRpb24uCi0gKipgc3BsaXRgIGlzIGEgY29sdW1uKiogYW5kIHRoZSBmaWxlcyBhcmUgb3JnYW5pc2VkIGludG8gYDxzdWJzZXQ+L3t0cmFpbix2YWwsdGVzdH0vYCwgc28KICBgbG9hZF9kYXRhc2V0KC4uLiwgc3BsaXQ9Li4uKWAgYW5kIHRoZSB2aWV3ZXIncyBzcGxpdCB0YWJzIHdvcmsuCi0gKipQZXItbW9kZWwgc3Vic2V0cyoqIGFyZSBzZWxlY3RhYmxlIGRpcmVjdGx5IGluIHRoZSB2aWV3ZXIgLyBsb2FkZXIuCi0gKipDYXB0aW9uIHByb3ZlbmFuY2UgYWRkZWQqKjogYHJhd19jYXB0aW9uYCwgYGNhcHRpb25fbl90b2tlbnNgLCBgY2FwdGlvbl9tb2RlbGAsIHBsdXMgdGhlCiAgb3JpZ2luYWwgQkxJUC0yIGNhcHRpb24gc2hhcmRzIHVuZGVyIGBjYXB0aW9ucy9gLgoKIyMgR2VuZXJhdG9ycwoKfCBrZXkgfCBtb2RlbCBpZCB8IG5hdGl2ZSByZXMgfCBzdGVwcyB8IGd1aWRhbmNlIHwKfC0tLS0tfC0tLS0tLS0tLS18LS0tLS0tLS0tLS18LS0tLS0tLXwtLS0tLS0tLS0tfAp8IGBzZDE1YCB8IGBzdGFibGUtZGlmZnVzaW9uLXYxLTUvc3RhYmxlLWRpZmZ1c2lvbi12MS01YCB8IDUxMiB8IDIwIHwgNy4wIHwKfCBgc2R4bGAgfCBgc3RhYmlsaXR5YWkvc3RhYmxlLWRpZmZ1c2lvbi14bC1iYXNlLTEuMGAgfCAxMDI0IHwgOCB8IDcuMCB8CnwgYGZsdXhfc2NobmVsbGAgfCBgYmxhY2stZm9yZXN0LWxhYnMvRkxVWC4xLXNjaG5lbGxgIHwgMTAyNCB8IDQgfCAwLjAgfAp8IGBrYW5kaW5za3kyMmAgfCBga2FuZGluc2t5LWNvbW11bml0eS9rYW5kaW5za3ktMi0yLWRlY29kZXJgIHwgNzY4IHwgMjUgfCA0LjAgfAp8IGBwaXhhcnRfc2lnbWFgIHwgYFBpeEFydC1hbHBoYS9QaXhBcnQtU2lnbWEtWEwtMi0xMDI0LU1TYCB8IDEwMjQgfCAyMCB8IDQuNSB8CnwgYHd1ZXJzdGNoZW5gIHwgYHdhcnAtYWkvd3VlcnN0Y2hlbmAgfCAxMDI0IHwgWzIwLCAxMF0gfCBbNC4wLCAwLjBdIHwKCiMjIEhvdyB0aGUgcGFpcnMgYXJlIGJ1aWx0CgpSZWFsIGltYWdlcyBhcmUgY2FwdGlvbmVkIHdpdGggKipTYWxlc2ZvcmNlL2JsaXAyLW9wdC0yLjdiKiosIHRoZSBjYXB0aW9uIGlzIGNsZWFuZWQgYW5kCmNhcHBlZCB0byA3NSBDTElQIHRva2VucywgYW5kIHRoYXQgKipleGFjdCBjYXB0aW9uIGlzIHJldXNlZCBieSBhbGwgc2l4IGdlbmVyYXRvcnMqKiBhbmQKYWxzbyBzdG9yZWQgb24gdGhlIHJlYWwgcm93LiBgc291cmNlX3JlYWxfaWRgIGxpbmtzIGV2ZXJ5IEFJIGltYWdlIGJhY2sgdG8gaXRzIHJlYWwKcGFydG5lciAoYHNvdXJjZV9yZWFsX2lkID09IGltYWdlX2lkYCBmb3IgcmVhbCByb3dzKS4KCiMjIENhbm9uaWNhbCBwcmVwcm9jZXNzIChsZWFrLWZyZWUpCgpCb3RoIHJlYWwgYW5kIEFJIGltYWdlcyBwYXNzIHRocm91Z2ggdGhlICoqc2FtZSoqIHBpcGVsaW5lOgoKMS4gRVhJRi10cmFuc3Bvc2Ug4oaSIGNvbnZlcnQgdG8gUkdCIOKGkiBjZW50ZXItY3JvcCB0byBzcXVhcmUg4oaSIExhbmN6b3MgcmVzaXplIHRvIDUxMi4KMi4gSlBFRy1lcXVhbGlzZSAocXVhbGl0eSA5NSwgNDo0OjQpIOKGkiBzYXZlIGFzIFBORyAoY29tcHJlc3MgbGV2ZWwgNiwgbm8gRVhJRi9JQ0MpLgoKQXBwbHlpbmcgYW4gaWRlbnRpY2FsIHRyYW5zZm9ybSB0byBib3RoIGNsYXNzZXMgcmVtb3ZlcyB0aGUgcmVzb2x1dGlvbiwgY29sb3VyLXNwYWNlIGFuZApKUEVHLWFydGVmYWN0IHNob3J0Y3V0cyB0aGF0IHdvdWxkIG90aGVyd2lzZSBsZXQgYSBtb2RlbCBjaGVhdC4KCiMjIFNwbGl0cwoKRGV0ZXJtaW5pc3RpYyAqKnBhaXItbGV2ZWwqKiBzcGxpdCBrZXllZCBvbiBgc291cmNlX3JlYWxfaWRgIChzZWVkIGA0MmApOiBhIHJlYWwgaW1hZ2UgYW5kCmFsbCBzaXggb2YgaXRzIEFJIHBhcnRuZXJzIGFsd2F5cyBsYW5kIGluIHRoZSBzYW1lIGZvbGQsIHNvIHRoZXJlIGlzICoqbm8gc2NlbmUgbGVha2FnZSoqCmJldHdlZW4gdHJhaW4gLyB2YWxpZGF0aW9uIC8gdGVzdC4KCnwgc3BsaXQgfCBpbWFnZXMgfAp8LS0tLS0tLXwtLS0tLS0tLXwKfCB0cmFpbiB8IDQ5LDM5MiB8CnwgdmFsaWRhdGlvbiB8IDEwLDEyMiB8CnwgdGVzdCB8IDEwLDQ4NiB8CgojIyBEYXRhc2V0IHN0cnVjdHVyZQoKYGBgCmFpLWltYWdlLWRldGVjdGlvbi1kYXRhc2V0LwrilJzilIDilIAgcmVhbC8gICAgICAgICAgICB7dHJhaW4sdmFsLHRlc3R9LyoucGFycXVldCAgICAjIDEwayByZWFsIHBob3RvcywgbGFiZWwgMArilJzilIDilIAgc2QxNS8gICAgICAgICAgICB7dHJhaW4sdmFsLHRlc3R9LyoucGFycXVldCAgICAjIDEwayBBSSBwZXIgZ2VuZXJhdG9yLCBsYWJlbCAxCuKUnOKUgOKUgCBzZHhsLyAgICAgICAgICAgIHt0cmFpbix2YWwsdGVzdH0vKi5wYXJxdWV0CuKUnOKUgOKUgCBmbHV4X3NjaG5lbGwvICAgIHt0cmFpbix2YWwsdGVzdH0vKi5wYXJxdWV0CuKUnOKUgOKUgCBrYW5kaW5za3kyMi8gICAgIHt0cmFpbix2YWwsdGVzdH0vKi5wYXJxdWV0CuKUnOKUgOKUgCBwaXhhcnRfc2lnbWEvICAgIHt0cmFpbix2YWwsdGVzdH0vKi5wYXJxdWV0CuKUnOKUgOKUgCB3dWVyc3RjaGVuLyAgICAgIHt0cmFpbix2YWwsdGVzdH0vKi5wYXJxdWV0CuKUnOKUgOKUgCBjYXB0aW9ucy8gICAgICAgIGNhcHRpb25zLSoucGFycXVldCAgICAgICAgICAgICMgb3JpZ2luYWwgQkxJUC0yIGNhcHRpb24gc2hhcmRzIChwcm92ZW5hbmNlKQrilJzilIDilIAgbWV0YWRhdGEvCuKUgiAgIOKUnOKUgOKUgCBtYW5pZmVzdC5wYXJxdWV0ICAgICAgICAjIGlkLCBwYWlyaW5nIGtleSwgbGFiZWwsIGdlbmVyYXRvciwgc3BsaXQgKG5vIGltYWdlIGJ5dGVzKQrilIIgICDilJzilIDilIAgc3BsaXRzLnBhcnF1ZXQgICAgICAgICAgIyBzb3VyY2VfcmVhbF9pZCAtPiBzcGxpdArilIIgICDilJzilIDilIAgY2FwdGlvbnMucGFycXVldCAgICAgICAgIyBjb25zb2xpZGF0ZWQgcmVhbC1pbWFnZSBjYXB0aW9ucyArIHByb3ZlbmFuY2UK4pSCICAg4pSc4pSA4pSAIGNvbmZpZy5qc29uICAgICAgICAgICAgICMgYnVpbGQgY29uZmlndXJhdGlvbiAoc2VlZHMsIG1vZGVscywgcGFyYW1zKQrilIIgICDilJTilIDilIAgdmFsaWRhdGlvbl9yZXBvcnQuanNvbiAgIyBsZWFrLWF1ZGl0IC8gc25pZmYtdGVzdCByZXN1bHRzCuKUnOKUgOKUgCBhaV9kZXRlY3RfY29tbW9uLnB5ICAgICAgICAgIyBzaGFyZWQgcHJlcHJvY2VzcyArIHNjaGVtYSBsaWJyYXJ5CuKUlOKUgOKUgCBSRUFETUUubWQKYGBgCgojIyBVc2FnZQoKYGBgcHl0aG9uCmZyb20gZGF0YXNldHMgaW1wb3J0IGxvYWRfZGF0YXNldAoKIyBldmVyeXRoaW5nLCBieSBzcGxpdApkcyA9IGxvYWRfZGF0YXNldCgiU2hhbm11azQ2MjIvYWktaW1hZ2UtZGV0ZWN0aW9uLWRhdGFzZXQiKSAgICAgICAgICAjIGRlZmF1bHQgc3Vic2V0CnRyYWluLCB2YWwsIHRlc3QgPSBkc1sidHJhaW4iXSwgZHNbInZhbGlkYXRpb24iXSwgZHNbInRlc3QiXQoKZXggPSB0cmFpblswXQpleFsiaW1hZ2UiXSAgICAgIyBQSUwuSW1hZ2UgKGRlY29kZWQgYXV0b21hdGljYWxseSwgcmVuZGVycyBpbiB0aGUgdmlld2VyKQpleFsibGFiZWwiXSAgICAgIyAwID0gcmVhbCwgMSA9IEFJCmV4WyJwcm9tcHQiXSAgICAjIHNoYXJlZCBjYXB0aW9uIChub3cgcHJlc2VudCBvbiByZWFsIHJvd3MgdG9vKQpleFsiZ2VuZXJhdG9yIl0gIyAncmVhbCcgb3Igb25lIG9mIHRoZSA2IGdlbmVyYXRvcnMKZXhbInNwbGl0Il0gICAgICMgJ3RyYWluJyB8ICd2YWxpZGF0aW9uJyB8ICd0ZXN0JwoKIyBhIHNpbmdsZSBtb2RlbCdzIHN1YnNldCAocmVhbCArIG9uZSBnZW5lcmF0b3IgYXJlIHNlcGFyYXRlIHN1YnNldHMpCmZsdXhfdGVzdCA9IGxvYWRfZGF0YXNldCgiU2hhbm11azQ2MjIvYWktaW1hZ2UtZGV0ZWN0aW9uLWRhdGFzZXQiLCAiZmx1eF9zY2huZWxsIiwgc3BsaXQ9InRlc3QiKQpyZWFsX3RyYWluID0gbG9hZF9kYXRhc2V0KCJTaGFubXVrNDYyMi9haS1pbWFnZS1kZXRlY3Rpb24tZGF0YXNldCIsICJyZWFsIiwgc3BsaXQ9InRyYWluIikKYGBgCgojIyBTY2hlbWEKCnwgY29sdW1uIHwgdHlwZSB8IGRlc2NyaXB0aW9uIHwKfC0tLS0tLS0tfC0tLS0tLXwtLS0tLS0tLS0tLS0tfAp8IGBpbWFnZWAgfCBJbWFnZSBgc3RydWN0e2J5dGVzLCBwYXRofWAgfCBjYW5vbmljYWwgNTEyw5c1MTIgUkdCIFBORyB8CnwgYGltYWdlX2lkYCB8IHN0cmluZyB8IGdsb2JhbGx5IHVuaXF1ZSwgZS5nLiBgcmVhbF8wMDAwMDFgIC8gYHNkeGxfMDAwMDAxYCB8CnwgYHNvdXJjZV9yZWFsX2lkYCB8IHN0cmluZyB8IHBhaXJpbmcga2V5OyBlcXVhbHMgYGltYWdlX2lkYCBmb3IgcmVhbCByb3dzIHwKfCBgbGFiZWxgIHwgaW50OCB8IDAgPSByZWFsLCAxID0gQUkgfAp8IGBnZW5lcmF0b3JgIHwgc3RyaW5nIHwgYHJlYWxgLCBgc2QxNWAsIGBzZHhsYCwgYGZsdXhfc2NobmVsbGAsIGBrYW5kaW5za3kyMmAsIGBwaXhhcnRfc2lnbWFgLCBgd3VlcnN0Y2hlbmAgfAp8IGBzb3VyY2VfZGF0YXNldGAgfCBzdHJpbmcgfCBgY29jb2AsIGBpbWFnZW5ldGAsIG9yIGA8Z2VuZXJhdG9yPmAgfAp8IGBzcGxpdGAgfCBzdHJpbmcgfCBgdHJhaW5gIC8gYHZhbGlkYXRpb25gIC8gYHRlc3RgIHwKfCBgcHJvbXB0YCB8IHN0cmluZyB8IHNoYXJlZCBpbWFnZS1ncm91bmRlZCBjYXB0aW9uICgqKm5vdyBwb3B1bGF0ZWQgZm9yIHJlYWwgcm93cyoqKSB8CnwgYHJhd19jYXB0aW9uYCB8IHN0cmluZyB8IHVuY2xlYW5lZCBCTElQLTIgb3V0cHV0IHwKfCBgY2FwdGlvbl9uX3Rva2Vuc2AgfCBpbnQxNiB8IENMSVAtdG9rZW4gbGVuZ3RoIG9mIGBwcm9tcHRgIHwKfCBgY2FwdGlvbl9tb2RlbGAgfCBzdHJpbmcgfCBjYXB0aW9uZXIgdXNlZCAoYFNhbGVzZm9yY2UvYmxpcDItb3B0LTIuN2JgKSB8CnwgYHdpZHRoYCAvIGBoZWlnaHRgIHwgaW50MTYgfCBhbHdheXMgNTEyIHwKfCBgb3JpZ193aWR0aGAgLyBgb3JpZ19oZWlnaHRgIHwgaW50MzIgfCBwcm92ZW5hbmNlIG9ubHkg4oCUICoqbmV2ZXIgdXNlIGFzIGEgdHJhaW5pbmcgZmVhdHVyZSoqIHwKfCBgZ2VuX21vZGVsX2lkYCB8IHN0cmluZyB8IEhGIG1vZGVsIGlkIChudWxsIGZvciByZWFsKSB8CnwgYGdlbl9zdGVwc2AgLyBgZ2VuX2d1aWRhbmNlYCAvIGBnZW5fbmF0aXZlX3Jlc2AgfCBpbnQxNiAvIGZsb2F0MzIgLyBpbnQxNiB8IGdlbmVyYXRpb24gcGFyYW1zIChudWxsIGZvciByZWFsKSB8CnwgYHNlZWRgIHwgaW50NjQgfCBgaGFzaChtYXN0ZXJfc2VlZCwgZ2VuZXJhdG9yLCBzb3VyY2VfcmVhbF9pZClgIChudWxsIGZvciByZWFsKSB8CnwgYHBpcGVsaW5lX3ZlcnNpb25gIHwgc3RyaW5nIHwgYDEuMmAgfAp8IGBzaGEyNTZgIHwgc3RyaW5nIHwgaGV4IGRpZ2VzdCBvZiB0aGUgUE5HIGJ5dGVzIHwKfCBgY3JlYXRlZF91dGNgIHwgc3RyaW5nIHwgSVNPLTg2MDEgVVRDIHRpbWVzdGFtcCB8CgojIyBMZWFrIGF1ZGl0CgpBICJzbmlmZiB0ZXN0IiB0cmFpbnMgYSBsaWdodHdlaWdodCBjbGFzc2lmaWVyIG9uIGVhY2ggZ2VuZXJhdG9yIHZzLiByZWFsIHVzaW5nIG9ubHkKbG93LWxldmVsIHN0YXRpc3RpY3MgYW5kIGNoZWNrcyBzZXBhcmFiaWxpdHkgc3RheXMgaGVhbHRoeSAoYDwwLjg1YCBoZWFsdGh5LApgMC44NeKAkzAuOTVgIGludmVzdGlnYXRlLCBgPjAuOTVgIGxlYWspOgoKfCBnZW5lcmF0b3IgfCBzY29yZSB8CnwtLS0tLS0tLS0tLXwtLS0tLS0tfAp8IHNkMTUgfCAwLjkxMCB8Cnwgc2R4bCB8IDAuOTczIHwKfCBmbHV4X3NjaG5lbGwgfCAwLjkxNyB8Cnwga2FuZGluc2t5MjIgfCAwLjk1OCB8CnwgcGl4YXJ0X3NpZ21hIHwgMC45NDMgfAp8IHd1ZXJzdGNoZW4gfCAwLjg0MyB8CnwgKipvdmVyYWxsKiogfCAqKjAuODYwKiogfAoKRnVsbCByZXN1bHRzIGluIGBtZXRhZGF0YS92YWxpZGF0aW9uX3JlcG9ydC5qc29uYC4KCiMjIFByb3ZlbmFuY2UsIGxpY2Vuc2UgYW5kIGludGVuZGVkIHVzZQoKLSAqKkltYWdlTmV0KiogY29udGVudCBpcyAqKm5vbi1jb21tZXJjaWFsIHJlc2VhcmNoIG9ubHkqKiAoSUxTVlJDIHRlcm1zKS4KLSAqKkNPQ08qKiBpbWFnZXMgYXJlIEZsaWNrci1zb3VyY2VkIChDcmVhdGl2ZSBDb21tb25zKS4KLSBBSSBpbWFnZXMgYXJlIHN5bnRoZXRpYyBvdXRwdXRzIG9mIHRoZSBtb2RlbHMgbGlzdGVkIGFib3ZlLCBlYWNoIHVuZGVyIGl0cyBvd24gbGljZW5zZS4KLSBUaGlzIGRhdGFzZXQgaW5oZXJpdHMgdGhlICoqbW9zdCByZXN0cmljdGl2ZSoqIGFwcGxpY2FibGUgdGVybTogKipub24tY29tbWVyY2lhbAogIHJlc2VhcmNoIHVzZSoqLiAqTm90IGxlZ2FsIGFkdmljZS4qCgojIyBMaW1pdGF0aW9ucyBhbmQgY2F2ZWF0cwoKLSBDb3JlIHNldCBpcyAqKnRleHQtdG8taW1hZ2Ugb25seSoqIChubyBpbWcyaW1nIC8gcmVmZXJlbmNlIGNvbmRpdGlvbmluZyk7IGRldGVjdG9ycwogIHRyYWluZWQgaGVyZSB0YXJnZXQgdGhlIHRleHQtdG8taW1hZ2UgdGhyZWF0IG1vZGVsLgotIENhcHRpb24gcXVhbGl0eSAoQkxJUC0yIGNvbmNpc2Ugc2VudGVuY2VzKSBib3VuZHMgcHJvbXB0IGRpdmVyc2l0eS4KLSBGTFVYLjEtc2NobmVsbCBhbmQgV8O8cnN0Y2hlbiBydW4gd2l0aCBDUFUgb2ZmbG9hZCBvbiBUNDsgZ2VuZXJhdGlvbiBjb25kaXRpb25zIGFyZQogIG90aGVyd2lzZSBjb25zaXN0ZW50IHdpdGggc3RhbmRhcmQgaW5mZXJlbmNlIHNldHRpbmdzLgotIFBlci1tb2RlbCBzdGVwIGNvdW50cyB3ZXJlIHJlZHVjZWQgZm9yIHNwZWVkIChTRFhMIDggc3RlcHMsIFNEIDEuNSAyMCBzdGVwcyksIHdoaWNoIGlzCiAgcmVwcmVzZW50YXRpdmUgb2YgcmVhbC13b3JsZCBmYXN0IGluZmVyZW5jZS4KCiMjIENpdGF0aW9uCgpgYGBiaWJ0ZXgKQG1pc2N7YWlpbWFnZV9kZXRlY3Rpb25fZGF0YXNldCwKICB0aXRsZSAgPSB7QUktSW1hZ2UgRGV0ZWN0aW9uIERhdGFzZXR9LAogIGF1dGhvciA9IHtTaGFubXVrNDYyMn0sCiAgeWVhciAgID0gezIwMjV9LAogIGhvd3B1Ymxpc2hlZCA9IHtcdXJse2h0dHBzOi8vaHVnZ2luZ2ZhY2UuY28vZGF0YXNldHMvU2hhbm11azQ2MjIvYWktaW1hZ2UtZGV0ZWN0aW9uLWRhdGFzZXR9fQp9CmBgYAo=").decode()
    readme="---\n"+yaml.safe_dump(front,sort_keys=False,allow_unicode=True)+"---\n\n"+body
    STATE["readme_done"]=True
    api.create_commit(DST_REPO,repo_type=REPO_TYPE,commit_message="build: dataset card",
        operations=[CommitOperationAdd("README.md",readme.encode()),
                    CommitOperationAdd(STATE_PATH,json.dumps(STATE,indent=2).encode())])
    print("pushed dataset card (",len(configs),"configs )")

In [ ]:
# 8) MAIN LOOP — convert + enrich (commit local) ; push on triggers ----
all_src=list_image_shards()
todo=[s for s in all_src if s not in set(STATE["pushed_sources"])]
print(f"source shards: {len(all_src)} total | {len(todo)} remaining")
PUSHER.last_push=time.time()
done_run,failed,tot_ms,tot_mc=0,[],0,0
try:
    for i,src in enumerate(todo,1):
        t0=time.time()
        try:
            staged,rows,ms,mc=convert_source(src)
            PUSHER.commit_local(src,staged,rows); done_run+=1; tot_ms+=ms; tot_mc+=mc
            print(f"[{i}/{len(todo)}] {src} -> {subset_of(src)}/  ({time.time()-t0:.1f}s)")
        except Exception as e:
            failed.append(src); print(f"[{i}/{len(todo)}] FAILED {src}: {type(e).__name__}: {e}"); traceback.print_exc()
        if PUSHER.due(): PUSHER.push("interval")
    PUSHER.push("final",force=True)
except KeyboardInterrupt:
    print("interrupted — staged batch already pushed by signal handler."); raise
print(f"\nconverted this run: {done_run} | failed: {len(failed)} | rows w/o split: {tot_ms} | rows w/o caption: {tot_mc}")
if failed: print("failed (retried next run):",failed[:20])

In [ ]:
# 9) verify + finish ---------------------------------------------------
STATE,_=load_state(); all_src=list_image_shards()
remaining=[s for s in all_src if s not in set(STATE["pushed_sources"])]
print(f"pushed source shards: {len(STATE['pushed_sources'])}/{len(all_src)} | remaining: {len(remaining)}")
files=api.list_repo_files(DST_REPO,repo_type=REPO_TYPE)
subs=sorted({f.split('/')[0] for f in files if '/train/' in f or '/val/' in f or '/test/' in f})
print("subsets present:",subs)
probe=next((f for f in files if "/test/" in f and f.endswith(".parquet")), None)
if remaining:
    print("Not finished — Run All again to resume from the remote ledger.")
elif probe is None:
    print("WARNING: ledger says done but no <subset>/<split>/*.parquet files exist.")
    print("Set RESET_BUILD=True in cell 2 and Run All once to rebuild.")
else:
    lp=hf_hub_download(DST_REPO,probe,repo_type=REPO_TYPE,token=TOKEN,force_download=True)
    sc=pq.read_schema(lp); it=sc.field("image").type
    ok=pa.types.is_struct(it) and {f.name for f in it}>={"bytes","path"}
    print("probe:",probe,"| image struct(bytes,path):",ok,"| has prompt&split:",{"prompt","split"}<=set(sc.names))
    t=pq.read_table(lp,columns=["generator","prompt"])
    reals=[p for gg,p in zip(t.column("generator").to_pylist(),t.column("prompt").to_pylist()) if gg=="real"]
    if reals: print("real rows in probe:",len(reals),"| with non-empty prompt:",sum(1 for p in reals if p))
    print("\nDATASET BUILT OK. Make public when ready:")
    print("  HfApi(token=TOKEN).update_repo_settings('%s',repo_type='dataset',private=False)"%DST_REPO)
    print("  https://huggingface.co/datasets/%s"%DST_REPO)